<a href="https://colab.research.google.com/github/pepealania/agentic-rag/blob/main/experiments/01_reproducibility_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 — Imports

from pathlib import Path
from datetime import datetime
import json
import platform
import shutil
import sys

import yaml

In [2]:
# Cell 2 — Project paths

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "default.yaml"

assert CONFIG_PATH.exists(), f"Configuration not found: {CONFIG_PATH}"

print(f"Project root: {PROJECT_ROOT}")
print(f"Configuration: {CONFIG_PATH}")

AssertionError: Configuration not found: /configs/default.yaml

In [ ]:
# Cell 3 — Load configuration

with CONFIG_PATH.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

assert isinstance(config, dict), "Configuration must be a YAML mapping."

print(yaml.safe_dump(config, sort_keys=False))

In [ ]:
# Cell 4 — Validate required configuration

required_sections = [
    "experiment",
    "model",
    "embeddings",
    "chunking",
    "retrieval",
    "pipeline",
    "evaluation",
    "tracing",
    "paths",
]

missing_sections = [
    section for section in required_sections
    if section not in config
]

assert not missing_sections, (
    f"Missing configuration sections: {missing_sections}"
)

assert config["retrieval"]["top_k"] > 0
assert config["pipeline"]["max_iterations"] > 0
assert config["chunking"]["chunk_size"] > 0
assert config["chunking"]["chunk_overlap"] >= 0
assert config["chunking"]["chunk_overlap"] < config["chunking"]["chunk_size"]
assert 0.0 <= config["model"]["temperature"] <= 2.0

print("Configuration validation: OK")

In [ ]:
# Cell 5 — Create experiment ID

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = config["experiment"]["name"]

experiment_id = f"{timestamp}_{experiment_name}"

print(f"Experiment ID: {experiment_id}")

In [ ]:
# Cell 6 — Create experiment output directory

outputs_root = PROJECT_ROOT / config["paths"]["outputs"]
experiment_dir = outputs_root / experiment_id

experiment_dir.mkdir(parents=True, exist_ok=False)

print(f"Experiment directory created:")
print(experiment_dir)

In [ ]:
# Cell 7 — Save the exact configuration used

config["experiment"]["id"] = experiment_id

experiment_config_path = experiment_dir / "config.yaml"

with experiment_config_path.open("w", encoding="utf-8") as file:
    yaml.safe_dump(
        config,
        file,
        sort_keys=False,
        allow_unicode=True,
    )

print(f"Configuration saved to: {experiment_config_path}")

In [ ]:
# Cell 8 — Record execution environment

environment = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "python_executable": sys.executable,
}

environment_path = experiment_dir / "environment.json"

with environment_path.open("w", encoding="utf-8") as file:
    json.dump(
        environment,
        file,
        indent=2,
    )

print(f"Environment information saved to: {environment_path}")

In [ ]:
# Cell 9 — Reproducibility summary

summary = {
    "experiment_id": experiment_id,
    "experiment_name": config["experiment"]["name"],
    "seed": config["experiment"]["seed"],
    "model": config["model"]["name"],
    "embeddings": config["embeddings"]["name"],
    "temperature": config["model"]["temperature"],
    "chunk_size": config["chunking"]["chunk_size"],
    "chunk_overlap": config["chunking"]["chunk_overlap"],
    "top_k": config["retrieval"]["top_k"],
    "max_iterations": config["pipeline"]["max_iterations"],
}

print(json.dumps(summary, indent=2))

In [ ]:
# Cell 10 — Verify reproducibility artifacts

expected_files = [
    experiment_config_path,
    environment_path,
]

for path in expected_files:
    assert path.exists(), f"Missing artifact: {path}"

print("Reproducibility setup: PASSED")
print(f"Experiment: {experiment_id}")
print(f"Artifacts: {experiment_dir}")